# 0824_lsw_004_imbalance_handling

검사유형별 5분리 구조(`0824_lsw_003_structure_comparison`에서 채택) 위에서 클래스 불균형 대응 기법을 비교합니다. 데이터 전처리와 평가 함수는 003과 동일하게 재사용합니다.

이번 노트북은 두 부분으로 구성됩니다: (1) SMOTE/ADASYN 적용 전에 소수 클래스(불량)에 이상치가 얼마나 있는지 가볍게 확인, (2) 불균형 처리 기법별 비교.

## 1. 설정과 라이브러리

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

EXPERIMENT_ID = "0824_lsw_004_imbalance_handling"
RANDOM_STATE = 42
DATA_PATH = Path("../data/raw/dataset.csv")
TARGET = "class"
TIME_COLUMN = "timestamp"
RECORD_ID = "record_id"
MODEL_DIR = Path("../models")

COST_SCENARIOS = {"1:10": (1, 10), "1:100": (1, 100)}

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
print("experiment:", EXPERIMENT_ID)

experiment: 0824_lsw_004_imbalance_handling


## 2. 데이터 로딩·전처리 (003과 동일)

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")
assert raw_df[RECORD_ID].is_unique, "record_id가 고유하지 않습니다."

dedup_columns = [c for c in raw_df.columns if c not in {RECORD_ID, TIME_COLUMN}]
duplicate_mask = raw_df.duplicated(subset=dedup_columns, keep="first")
clean_df = raw_df.loc[~duplicate_mask].copy().reset_index(drop=True)

clean_df[TIME_COLUMN] = pd.to_datetime(clean_df[TIME_COLUMN], errors="raise", utc=True)
clean_df = clean_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

feature_columns_all = [c for c in clean_df.columns if c not in {RECORD_ID, TIME_COLUMN, TARGET}]

timestamps = clean_df[TIME_COLUMN]
timestamp_group_sizes = timestamps.value_counts(sort=False).sort_index()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
train_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.60, side="left"))]
valid_end_time = timestamp_group_sizes.index[int(np.searchsorted(cumulative_rows, len(clean_df) * 0.80, side="left"))]

train_mask = timestamps <= train_end_time
valid_mask = (timestamps > train_end_time) & (timestamps <= valid_end_time)
test_mask = timestamps > valid_end_time

print("rows_after_dedup:", len(clean_df))
pd.DataFrame(
    [
        {"split": name, "rows": int(mask.sum())}
        for name, mask in [("train", train_mask), ("validation", valid_mask), ("test", test_mask)]
    ]
).set_index("split")

rows_after_dedup: 391992


,rows
split,
train,235222
validation,78374
test,78396


## 3. 평가 함수 (003과 동일)

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fp, cost_fn):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp


def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    candidates = np.sort(np.unique(proba_val))[::-1]
    for t in candidates:
        y_pred = (proba_val >= t).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return float(t)
    return 0.0


def evaluate_at_threshold(y_true, proba, threshold):
    y_pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    result = {
        "threshold": threshold,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "slip_rate": slip_rate(y_true, y_pred),
        "volume_reduction": volume_reduction(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else float("nan"),
    }
    for name, (cost_fp, cost_fn) in COST_SCENARIOS.items():
        result[f"total_cost_{name}"] = total_cost(y_true, y_pred, cost_fp, cost_fn)
    return result


def get_non_constant_columns(candidate_columns, train_frame):
    nunique = train_frame[candidate_columns].nunique()
    return nunique[nunique > 1].index.tolist()

## 4. 검사유형별 subset 준비

`0824_lsw_003`에서 채택한 구조(검사유형별 5분리, 유형별 임계값)를 이번 실험의 baseline으로 그대로 재현합니다.

In [4]:
type_splits = {}
for inspection_type in sorted(clean_df["inspection_type"].unique()):
    type_mask = clean_df["inspection_type"] == inspection_type
    type_train_df = clean_df.loc[train_mask & type_mask]
    type_valid_df = clean_df.loc[valid_mask & type_mask]
    type_test_df = clean_df.loc[test_mask & type_mask]
    type_feature_columns = get_non_constant_columns(feature_columns_all, type_train_df)
    type_splits[inspection_type] = {
        "train": type_train_df,
        "valid": type_valid_df,
        "test": type_test_df,
        "feature_columns": type_feature_columns,
    }
    print(f"type {inspection_type}: train={len(type_train_df)}, features={len(type_feature_columns)}, "
          f"train_pos={int((type_train_df[TARGET]==1).sum())}")

type 0: train=41961, features=47, train_pos=57
type 1: train=34041, features=34, train_pos=555


type 2: train=74910, features=24, train_pos=579


type 3: train=80792, features=22, train_pos=620
type 4: train=3518, features=24, train_pos=13


## 5. SMOTE/ADASYN 적용 전 — 소수 클래스(불량) 이상치 체크 (수정판)

**1차 시도의 결함**: 처음엔 정상+불량을 섞은 Train 전체로 IQR "정상 범위"를 구했는데, 불량 비율이 1~6%뿐이라 이 범위는 사실상 정상(다수) 클래스의 분포와 같다. 불량의 40~54%가 이 범위를 벗어난다는 결과는 "불량이 이상치"라는 뜻이 아니라 "불량과 정상의 분포가 원래 다르다(=오히려 좋은 신호)"는 뜻일 수 있다는 지적을 받았다.

**확인할 것 두 가지**:
1. 지금 기준(피처의 20% 이상이 범위 이탈)이 **정상 클래스에도 비슷하게 많이 걸리는지** — 그렇다면 기준 자체가 이 데이터에 비해 너무 헐거운 것이지, 불량이 특별히 이상한 게 아니다.
2. 임계값을 여러 단계로 바꿔가며 불량이 "뚝 떨어지는 절벽 구간"이 있는지 — 있다면 그 지점을 데이터 기반 임계값으로 쓴다.

방법을 두 가지 고친다: (a) IQR "정상 범위"는 **정상(class=0) 클래스만으로** 계산한다(불량이 섞여 범위가 넓어지는 것을 방지). (b) 고정된 20% 기준 하나만 보지 않고, 여러 임계값에서 정상 vs 불량의 flagged 비율을 나란히 표로 본다.

In [5]:
thresholds_to_check = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

outlier_scores_by_type = {}
cliff_rows = []

for inspection_type, split in type_splits.items():
    train_df = split["train"]
    feature_columns = split["feature_columns"]

    majority_df = train_df.loc[train_df[TARGET] == 0, feature_columns]
    minority_df = train_df.loc[train_df[TARGET] == 1, feature_columns]

    # 정상(class=0) 클래스만으로 "정상 운영 범위"를 정의
    q1 = majority_df.quantile(0.25)
    q3 = majority_df.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    def outlier_feature_fraction(df):
        is_outside = (df < lower) | (df > upper)
        return is_outside.mean(axis=1)

    majority_score = outlier_feature_fraction(majority_df)
    minority_score = outlier_feature_fraction(minority_df)
    outlier_scores_by_type[inspection_type] = {"majority": majority_score, "minority": minority_score}

    for th in thresholds_to_check:
        cliff_rows.append(
            {
                "inspection_type": inspection_type,
                "threshold": th,
                "정상_flagged_%": (majority_score > th).mean() * 100,
                "불량_flagged_%": (minority_score > th).mean() * 100,
            }
        )

cliff_df = pd.DataFrame(cliff_rows)
minority_pivot = cliff_df.pivot(index="threshold", columns="inspection_type", values="불량_flagged_%")
majority_pivot = cliff_df.pivot(index="threshold", columns="inspection_type", values="정상_flagged_%")

print("=== 정상(다수) 클래스: 임계값별 flagged 비율(%) — 기준 자체가 헐거운지 확인 ===")
display(majority_pivot.round(2))

print("\n=== 불량(소수) 클래스: 임계값별 flagged 비율(%) — 절벽 구간 확인 ===")
display(minority_pivot.round(2))

=== 정상(다수) 클래스: 임계값별 flagged 비율(%) — 기준 자체가 헐거운지 확인 ===


inspection_type,0,1,2,3,4
threshold,,,,,
0.00,76.67,95.65,76.16,76.03,44.11
0.05,65.54,75.00,54.38,45.05,27.56
0.10,58.33,45.19,35.58,29.31,17.66
0.15,41.84,27.78,19.27,16.38,11.81
0.20,33.63,21.51,12.62,7.20,9.59
0.25,23.85,13.73,2.89,1.68,4.19
0.30,11.23,6.82,0.27,0.40,2.51
0.40,4.35,0.00,0.00,0.00,0.83
0.50,0.36,0.00,0.00,0.00,0.03



=== 불량(소수) 클래스: 임계값별 flagged 비율(%) — 절벽 구간 확인 ===


inspection_type,0,1,2,3,4
threshold,,,,,
0.00,100.00,99.46,68.91,95.48,69.23
0.05,89.47,96.04,32.82,74.68,69.23
0.10,64.91,83.42,21.42,40.65,61.54
0.15,61.40,63.42,14.51,18.55,61.54
0.20,49.12,42.70,6.39,8.39,53.85
0.25,40.35,21.26,0.69,3.39,7.69
0.30,26.32,4.50,0.17,0.00,7.69
0.40,8.77,0.36,0.00,0.00,0.00
0.50,1.75,0.00,0.00,0.00,0.00


In [6]:
ratio_pivot = minority_pivot / majority_pivot.replace(0, np.nan)
print("=== 불량/정상 flagged 비율 (1.0 = 차이 없음, 클수록 불량이 진짜로 더 튄다는 뜻) ===")
ratio_pivot.round(2)

=== 불량/정상 flagged 비율 (1.0 = 차이 없음, 클수록 불량이 진짜로 더 튄다는 뜻) ===


inspection_type,0,1,2,3,4
threshold,,,,,
0.00,1.30,1.04,0.90,1.26,1.57
0.05,1.37,1.28,0.60,1.66,2.51
0.10,1.11,1.85,0.60,1.39,3.48
0.15,1.47,2.28,0.75,1.13,5.21
0.20,1.46,1.99,0.51,1.16,5.62
0.25,1.69,1.55,0.24,2.02,1.83
0.30,2.34,0.66,0.65,0.00,3.06
0.40,2.02,120.67,NaN,NaN,0.00
0.50,4.84,NaN,NaN,NaN,0.00


## 6. 결론 및 다음 단계 (1차 — 이상치 체크까지, 수정판)

### 1차 시도(섞인 IQR, 고정 20% 기준)가 과장됐음을 확인

정상(class=0) 클래스만으로 "정상 범위"를 다시 정의하고, 같은 20% 기준을 **정상 클래스에도** 적용해보니:

| inspection_type | 정상 flagged(th=0.2) | 불량 flagged(th=0.2) | 불량/정상 비율 |
|---|---:|---:|---:|
| 0 | 33.6% | 49.1% | 1.46 |
| 1 | 21.5% | 42.7% | 1.99 |
| 2 | 12.6% | 6.4% | 0.51 (불량이 오히려 덜 튐) |
| 3 | 7.2% | 8.4% | 1.16 |
| 4 | 9.6% | 53.9% | 5.62 (표본 13개, 노이즈 큼) |

**type0/1은 정상 클래스도 20~34%나 flagged된다** — 즉 이 기준 자체가 이 데이터에서 헐겁고, 1차 분석의 "49%/42.5%" 헤드라인 수치는 상당 부분 "불량이 이상해서"가 아니라 "기준이 느슨해서"였다.

### 절벽(cliff)은 뚜렷하지 않았지만, 임계값을 올리면 진짜 신호가 드러난다

정상/불량 flagged 곡선은 완만하게 감소해서 단일 절벽은 없었다. 대신 임계값별 **불량/정상 비율**을 보면:

- **type0**: 임계값을 올릴수록 비율이 1.1~1.7 → 2.3~4.8로 커진다. 즉 완만한 수준(th≈0.2)에서는 정상과 큰 차이가 없지만, **극단적인 수준(th≥0.3, 불량 26~40%만 해당)에서는 진짜로 불량이 더 튄다.** 진짜 이상치 후보는 49%가 아니라 26~40% 수준으로 좁혀야 한다.
- **type1**: 비율이 th=0.3에서 0.66으로 **1 밑으로 떨어진다** — 즉 극단적인 구간에서는 오히려 불량이 정상보다 덜 튄다. 42.5%라는 헤드라인은 실질적으로 이상치 신호가 아니라 대부분 기준의 느슨함이었다.
- **type2**: 모든 임계값에서 비율이 1 미만 — 불량이 정상보다 오히려 덜 극단적이다. 이상치 우려 없음.
- **type3**: 비율이 1.1~2.0 수준으로 약한 신호만 있다.
- **type4**: 표본이 13개뿐이라 결론을 내리기 어렵다.

### 다음 단계

- **불량 샘플을 대량(40~54%) 제거하는 건 근거가 부족하다** — 상당 부분 실제 신호(진짜 불량 패턴)를 지우는 것이 된다. 이 진단만으로 SMOTE 적용을 막을 이유는 없다.
- type0만 극단적인 임계값(th≥0.3, 실질 이상치 후보 26~40%)에서 유의미한 분리가 있으므로, type0에서 SMOTE가 잘 안 되면 이 서브셋을 살펴보는 정도로 남겨둔다.
- 나머지 유형(1/2/3/4)은 이상치 제거 없이 바로 SMOTE/ADASYN을 시도한다.
- 다음 셀부터 불균형 처리 기법(class_weight/scale_pos_weight, SMOTE, ADASYN, undersampling)을 유형별로 비교한다.


### 개념 정정 (추후 처리 예정)

위 "불량/정상 비율" 분석도 개념적으로 틀렸다는 지적을 받았다: 불량이 정상보다 극단적인 값을 갖는 건 노이즈가 아니라 **불량이라는 라벨 자체의 정상적인 신호**일 수 있다(오히려 분류를 쉽게 만듦). 진짜 이상치(측정 오류·오표기)는 "불량이라서 많은 것"이 아니라 **라벨과 무관하게 비슷한 비율로, 극소수만, 그 컬럼의 전체 분포 기준으로 튀는 것**이어야 한다. 이 기준으로 다시 보는 건 이후로 미루고, 이번엔 이상치 처리 없이 불균형 기법부터 비교한다.

## 7. 불균형 처리 기법 비교

검사유형별(5분리) 구조 위에서 baseline(리샘플링 없음) 대비 4가지 기법을 비교합니다: `class_weight`(scale_pos_weight), `SMOTE`, `ADASYN`, `Random Undersampling`. Validation/Test는 원본 그대로 두고, **Train에만** 리샘플링을 적용합니다(데이터 누수 방지). 임계값은 003의 결론대로 유형별 Validation 기준으로 선택합니다.

In [7]:
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import RandomUnderSampler


def build_model(scale_pos_weight=1.0):
    return XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
    )


def resample_train(technique, X_train, y_train, n_pos):
    k_neighbors = max(1, min(5, n_pos - 1))
    if technique == "smote":
        return SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors).fit_resample(X_train, y_train)
    if technique == "adasyn":
        try:
            return ADASYN(random_state=RANDOM_STATE, n_neighbors=k_neighbors).fit_resample(X_train, y_train)
        except ValueError:
            return X_train, y_train
    if technique == "undersample":
        return RandomUnderSampler(random_state=RANDOM_STATE).fit_resample(X_train, y_train)
    return X_train, y_train


imbalance_results = []
imbalance_models = {}
techniques = ["baseline", "class_weight", "smote", "adasyn", "undersample"]

for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]

    X_train, y_train = train_df[feature_columns], train_df[TARGET]
    X_valid, y_valid = valid_df[feature_columns], valid_df[TARGET]
    X_test, y_test = test_df[feature_columns], test_df[TARGET]

    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())
    scale_pos_weight = n_neg / n_pos

    for technique in techniques:
        if technique == "class_weight":
            X_res, y_res = X_train, y_train
            model = build_model(scale_pos_weight=scale_pos_weight)
        else:
            X_res, y_res = resample_train(technique, X_train, y_train, n_pos)
            model = build_model()

        model.fit(X_res, y_res)

        valid_proba = model.predict_proba(X_valid)[:, 1]
        threshold = select_threshold(y_valid, valid_proba)
        test_proba = model.predict_proba(X_test)[:, 1]

        result = evaluate_at_threshold(y_test, test_proba, threshold)
        result["inspection_type"] = inspection_type
        result["technique"] = technique
        result["n_train_resampled"] = len(y_res)
        result["n_train_pos_resampled"] = int((np.asarray(y_res) == 1).sum())
        imbalance_results.append(result)
        imbalance_models[(inspection_type, technique)] = model

imbalance_results_df = pd.DataFrame(imbalance_results)
imbalance_results_df.head()

,threshold,tn,fp,fn,tp,slip_rate,volume_reduction,pr_auc,roc_auc,total_cost_1:10,total_cost_1:100,inspection_type,technique,n_train_resampled,n_train_pos_resampled
0,6.372882e-06,8857,7794,22,111,0.165414,0.531920,0.067265,0.746373,8014,9994,0,baseline,41961,57
1,9.474958e-06,9512,7139,45,88,0.338346,0.571257,0.046449,0.675575,7589,11639,0,class_weight,41961,57
2,3.211504e-07,556,16095,9,124,0.067669,0.033391,0.029829,0.633109,16185,16995,0,smote,83808,41904
3,7.277359e-06,6699,9952,29,104,0.218045,0.402318,0.040245,0.676381,10242,12852,0,adasyn,83812,41908
4,2.814935e-01,10621,6030,40,93,0.300752,0.637860,0.054369,0.740493,6430,10030,0,undersample,114,57


In [8]:
detail_columns = ["inspection_type", "technique", "threshold", "tn", "fp", "fn", "tp", "pr_auc", "roc_auc", "slip_rate", "volume_reduction"]
imbalance_detail_df = imbalance_results_df[detail_columns].sort_values(["inspection_type", "technique"]).reset_index(drop=True)
imbalance_detail_df

,inspection_type,technique,threshold,tn,fp,fn,tp,pr_auc,roc_auc,slip_rate,volume_reduction
0,0,adasyn,7.277359e-06,6699,9952,29,104,0.040245,0.676381,0.218045,0.402318
1,0,baseline,6.372882e-06,8857,7794,22,111,0.067265,0.746373,0.165414,0.531920
2,0,class_weight,9.474958e-06,9512,7139,45,88,0.046449,0.675575,0.338346,0.571257
3,0,smote,3.211504e-07,556,16095,9,124,0.029829,0.633109,0.067669,0.033391
4,0,undersample,2.814935e-01,10621,6030,40,93,0.054369,0.740493,0.300752,0.637860
5,1,adasyn,4.003654e-05,571,9757,8,776,0.535932,0.888072,0.010204,0.055287
6,1,baseline,2.536666e-05,1869,8459,8,776,0.403144,0.875167,0.010204,0.180964
7,1,class_weight,1.266086e-05,338,9990,3,781,0.401299,0.861941,0.003827,0.032727
8,1,smote,5.872927e-05,2279,8049,11,773,0.481484,0.885938,0.014031,0.220662
9,1,undersample,5.340467e-03,4239,6089,12,772,0.251248,0.840760,0.015306,0.410438


## 8. 결과 비교 (시도 로그)

In [9]:
slip_pivot = imbalance_results_df.pivot(index="inspection_type", columns="technique", values="slip_rate")[techniques]
vol_pivot = imbalance_results_df.pivot(index="inspection_type", columns="technique", values="volume_reduction")[techniques]
cost10_pivot = imbalance_results_df.pivot(index="inspection_type", columns="technique", values="total_cost_1:10")[techniques]

print("=== Slip Rate ===")
display((slip_pivot * 100).round(2))
print("\n=== Volume Reduction ===")
display((vol_pivot * 100).round(2))
print("\n=== 총비용(1:10) ===")
display(cost10_pivot.round(0))

=== Slip Rate ===


technique,baseline,class_weight,smote,adasyn,undersample
inspection_type,,,,,
0,16.54,33.83,6.77,21.80,30.08
1,1.02,0.38,1.40,1.02,1.53
2,1.71,0.85,1.28,1.99,6.12
3,7.13,2.32,5.14,9.29,1.99
4,0.00,0.00,0.00,0.00,0.00



=== Volume Reduction ===


technique,baseline,class_weight,smote,adasyn,undersample
inspection_type,,,,,
0,53.19,57.13,3.34,40.23,63.79
1,18.10,3.27,22.07,5.53,41.04
2,18.93,2.59,7.98,12.21,45.20
3,32.35,12.52,35.45,43.46,9.67
4,0.00,0.00,0.00,0.27,2.05



=== 총비용(1:10) ===


technique,baseline,class_weight,smote,adasyn,undersample
inspection_type,,,,,
0,8014,7589,16185,10242,6430
1,8539,10020,8159,9837,6209
2,15078,18032,17067,16338,10541
3,20718,26374,19666,17515,27208
4,730,730,730,728,715


## 9. 결론 및 다음 단계 (2차 — 불균형 처리 기법 비교)

### 결과 요약

| type | baseline Slip | class_weight | smote | adasyn | undersample | Slip≤1% 만족 기법 |
|---|---:|---:|---:|---:|---:|---|
| 0 | 16.54% | 33.83% | 6.77% | 21.80% | 30.08% | 없음 |
| 1 | 1.02% | **0.38%** | 1.40% | 1.02% | 1.53% | class_weight, adasyn |
| 2 | 1.71% | **0.85%** | 1.28% | 1.99% | 6.12% | class_weight |
| 3 | 7.13% | 2.32% | 5.14% | 9.29% | 1.99% | 없음 |
| 4 | 0.00% | 0.00% | 0.00% | 0.00% | 0.00% | 전부(표본 부족으로 자명) |

### 핵심 관찰

- **모든 유형에 두루 이기는 기법은 없다.** 유형마다 반응이 다르다 — type0은 SMOTE가 Slip Rate를 가장 낮추지만(6.77%) Volume Reduction이 3.3%로 폭락한다. type3은 ADASYN이 Volume Reduction을 43.5%까지 올리지만 Slip Rate도 9.3%로 악화된다.
- **class_weight는 대체로 Slip Rate를 낮추지만(더 안전) Volume Reduction을 크게 희생시킨다** — type1(18.1%→3.3%), type2(18.9%→2.6%), type3(32.4%→12.5%) 전부 큰 폭으로 줄었다. `scale_pos_weight`가 모델을 "양성 쪽으로 더 민감하게" 만들면서 확률 보정이 흐트러져, 같은 Slip Rate 제약을 만족하려면 임계값을 훨씬 낮게(더 보수적으로) 잡아야 하는 것으로 보인다.
- **SMOTE/ADASYN은 유형별로 결과가 엇갈린다** — type1은 SMOTE가 Volume Reduction을 18.1%→22.1%로 개선하지만, type0은 SMOTE가 오히려 크게 악화(53.2%→3.3%)시킨다. type0의 이 악화는 지난 절에서 본 소수 클래스의 이질성(정상과 유사한 값 vs 극단적인 값이 섞여 있음)과 관련 있을 수 있다.
- **Undersampling은 type0/1/2에서 총비용(1:10)을 가장 크게 낮추지만, Slip Rate는 전부 목표를 벗어난다.** type0의 경우 이론상 최대 허용 Slip Rate(12절, 003 노트북)가 1:10에서 100%였으므로, 이 결과가 "비용 관점에서는" 정당화될 수 있다는 점은 참고할 만하다.
- type0/3/4는 어떤 기법으로도 Slip Rate ≤1%를 만족하지 못한다 — 불균형 처리만으로는 표본 부족/분리 어려움 문제가 완전히 해결되지 않는다.

### 다음 단계

- type1/2는 class_weight를 유력 후보로 남기되, Volume Reduction 손실이 크므로 실제 채택은 총비용 기준으로 재검토한다.
- type0/3/4는 이번 4가지 기법만으로는 해결이 안 됨 — feature selection(다음 노트북) 또는 이상치 재검토(추후)와 결합해봐야 한다.
- 각 기법의 정확한 정의와 관점 차이는 실험 문서에 정리한다.


## 10. 라벨 클렌징 (모순 라벨 제거)

`dev`에 병합된 `0824_dongjin_009/013/019`에서 가져온 아이디어입니다: Train 안에서 **피처 값이 완전히 동일한데 라벨(`class`)이 0과 1로 갈리는 행**을 찾아 제거합니다. 통계적 판단이 필요한 "이상치"와 달리, 이건 논리적으로 확실한 모순(같은 입력인데 다른 정답)이라 안전하게 제거할 수 있습니다.

리샘플링 없이(baseline 구조 그대로) 라벨 클렌징만 단독으로 적용해 효과를 분리해서 봅니다 — 한 번에 하나만 바꾸는 원칙을 유지합니다.

In [10]:
label_cleansing_results = []

for inspection_type, split in type_splits.items():
    train_df, valid_df, test_df = split["train"], split["valid"], split["test"]
    feature_columns = split["feature_columns"]

    nunique_classes = train_df.groupby(feature_columns)[TARGET].transform("nunique")
    contradictory_mask = nunique_classes > 1
    n_removed = int(contradictory_mask.sum())

    train_clean_df = train_df.loc[~contradictory_mask]

    X_valid, y_valid = valid_df[feature_columns], valid_df[TARGET]
    X_test, y_test = test_df[feature_columns], test_df[TARGET]

    model = build_model()
    model.fit(train_clean_df[feature_columns], train_clean_df[TARGET])

    valid_proba = model.predict_proba(X_valid)[:, 1]
    threshold = select_threshold(y_valid, valid_proba)
    test_proba = model.predict_proba(X_test)[:, 1]

    result = evaluate_at_threshold(y_test, test_proba, threshold)
    result["inspection_type"] = inspection_type
    result["technique"] = "label_cleansing"
    result["n_removed"] = n_removed
    result["n_train_resampled"] = len(train_clean_df)
    label_cleansing_results.append(result)

label_cleansing_df = pd.DataFrame(label_cleansing_results).set_index("inspection_type")
label_cleansing_df[["n_removed", "threshold", "tn", "fp", "fn", "tp", "pr_auc", "slip_rate", "volume_reduction"]]

,n_removed,threshold,tn,fp,fn,tp,pr_auc,slip_rate,volume_reduction
inspection_type,,,,,,,,,
0,24,1.315239e-05,11433,5218,46,87,0.036541,0.345865,0.686625
1,0,2.536666e-05,1869,8459,8,776,0.403144,0.010204,0.180964
2,28,5.985351e-07,420,18030,2,701,0.470404,0.002845,0.022764
3,0,6.984155e-06,9700,20288,43,560,0.269568,0.071310,0.323463
4,0,1.142581e-06,0,730,0,26,0.023284,0.000000,0.000000


In [11]:
baseline_df = imbalance_results_df[imbalance_results_df["technique"] == "baseline"].set_index("inspection_type")

label_cleansing_comparison = pd.DataFrame(
    {
        "n_removed(Train 중)": label_cleansing_df["n_removed"],
        "baseline_TN": baseline_df["tn"],
        "label_cleansing_TN": label_cleansing_df["tn"],
        "baseline_FN": baseline_df["fn"],
        "label_cleansing_FN": label_cleansing_df["fn"],
        "baseline_PR-AUC": baseline_df["pr_auc"],
        "label_cleansing_PR-AUC": label_cleansing_df["pr_auc"],
    }
)
label_cleansing_comparison["ΔTN"] = label_cleansing_comparison["label_cleansing_TN"] - label_cleansing_comparison["baseline_TN"]
label_cleansing_comparison["ΔFN"] = label_cleansing_comparison["label_cleansing_FN"] - label_cleansing_comparison["baseline_FN"]
label_cleansing_comparison

,n_removed(Train 중),baseline_TN,label_cleansing_TN,baseline_FN,label_cleansing_FN,baseline_PR-AUC,label_cleansing_PR-AUC,ΔTN,ΔFN
inspection_type,,,,,,,,,
0,24,8857,11433,22,46,0.067265,0.036541,2576,24
1,0,1869,1869,8,8,0.403144,0.403144,0,0
2,28,3492,420,12,2,0.356827,0.470404,-3072,-10
3,0,9700,9700,43,43,0.269568,0.269568,0,0
4,0,0,0,0,0,0.023284,0.023284,0,0


In [12]:
pooled_tn_baseline = int(baseline_df["tn"].sum())
pooled_fn_baseline = int(baseline_df["fn"].sum())
pooled_tn_cleansing = int(label_cleansing_df["tn"].sum())
pooled_fn_cleansing = int(label_cleansing_df["fn"].sum())

print(f"baseline 합계: TN={pooled_tn_baseline}, FN={pooled_fn_baseline}")
print(f"label_cleansing 합계: TN={pooled_tn_cleansing}, FN={pooled_fn_cleansing}")
print(f"ΔTN={pooled_tn_cleansing - pooled_tn_baseline}, ΔFN={pooled_fn_cleansing - pooled_fn_baseline}")

baseline 합계: TN=23918, FN=85
label_cleansing 합계: TN=23422, FN=99
ΔTN=-496, ΔFN=14


## 11. 결론 및 다음 단계 (3차 — 라벨 클렌징)

### 결과 — 전체(pooled) 기준이 핵심 결론

| | TN | FN |
|---|---:|---:|
| baseline | 23,918 | 85 |
| label_cleansing | 23,422 | 99 |
| **Δ** | **-496** | **+14** |

**전체 합산으로 보면 라벨 클렌징은 명확한 순손해다** — TN 496건 감소, FN
14건 증가로 둘 다 나빠졌다(dominated, 교환비를 따질 필요도 없음). 이는
`dev`의 dongjin 결과("pooled PR-AUC 0.323→0.348 개선")와 정반대 결론이다
— PR-AUC라는 threshold-무관 지표만 보면 개선처럼 보이지만, 실제 운영
관점(TN/FN 원본 건수)으로 보면 손해라는 게 이번 검증에서 드러났다.

### 유형별 상세

| type | 제거 건수 | ΔTN | ΔFN | baseline PR-AUC | label_cleansing PR-AUC |
|---|---:|---:|---:|---:|---:|
| 0 | 24 | +2,576 | +24 | 0.067 | **0.037 (하락)** |
| 1 | 0 | 0 | 0 | 0.403 | 0.403 (변화 없음) |
| 2 | 28 | -3,072 | -10 | 0.357 | **0.470 (상승)** — 그러나 임계값 급락으로 Volume Reduction 붕괴 |
| 3 | 0 | 0 | 0 | 0.270 | 0.270 (변화 없음) |
| 4 | 0 | 0 | 0 | 0.023 | 0.023 (변화 없음) |

전체 손해의 대부분은 **type0 하나에서 FN이 24건이나 늘어난 것**이 원인이다
— type2는 반대로 FN이 10건 줄었지만(더 안전해짐) 그 대가로 TN이
3,072건이나 줄어(덜 효율적) 상쇄되지 못했다. type0은 Train 불량이
57건뿐인데 24건(42%)이나 제거되면서, 노이즈가 아니라 "47개 피처만으로는
구분 안 되는 진짜 애매한 경계 사례"를 지웠을 가능성이 크다.

### 다음 단계

- 라벨 클렌징을 **모든 유형에 일괄 적용하는 건 전체 기준으로도 근거가
  없다** — 채택하지 않는다.
- type2는 FN이 줄어드는 방향(더 안전해짐)이라 안전 최우선 정책에서는
  선별 검토 여지가 있지만, 이번 범위에서는 더 파고들지 않는다.
- 이것으로 불균형 처리 단계(004)를 마무리하고, 다음은 feature selection
  (005, `mapping.json` 마스킹 포함)으로 넘어간다.
